# OpenMP complex compute constructs

# OpenMP complex compute constructs

In the previous lesson, **Single line constructs**, we made extensive use of the combined compute directive `!$omp target teams distribute parallel do`.
At the time, we discussed briefly what each clause in this directive was doing, and in this notebook we will explore what happens if we break this directive apart and combine the clauses in different ways.
We will then investigate how different approaches to parallelising nested loops affect the performance of the code.

Let us begin by checking that we have an appropriate GPU to run our code on using `rocm-smi`, moving into our work directory and making sure that our work environment is clean.

In [ ]:
rocm-smi
cd $HOME/AMD_OpenMP_Exercises/2c-OpenMP_complex_constructs/Fortran
make clean
export FC=amdflang

In this lesson, we want to see what the implementation is doing for each compute directive.
We can do this by setting the environment variable `LIBOMPTARGET_KERNEL_TRACE` to 1.
As the name suggests, this will print a trace for each OpenMP kernel launched at run time, with details of how it is executed.
We will see this in action soon - for now, let's set that variable.

In [ ]:
export LIBOMPTARGET_KERNEL_TRACE=1

## Breaking down compute constructs

Firstly, let's return to the full directive.
The code in [`saxpy_gpu_paralleldo.F90`](./Fortran/saxpy_gpu_paralleldo.F90) is the same as that in the previous lesson - that is to say, a simple saxpy code offloaded to the GPU using the `!$omp target teams distribute parallel do` directive.

Let's build and run it, and then we can break down the outputs:

In [ ]:
make saxpy_gpu_paralleldo
./saxpy_gpu_paralleldo

There is a lot of information here.
Let's break down the important outputs:
 - **DEVID** - the ID of the device the kernel is running on,
 - **teamsXthrds** - the number of teams generated, and how many threads are within each team,
 - **sgpr_count** - the number of scalar registers required by a wavefront,
 - **vgpr_count** - the number of vector registers required by each work-item,
 - **tripcount** - the number of loop iterations the runtime was asked to share out.

For the full combined directive, we see that 416 teams are generated with 256 threads each, and a vector register usage of 13.
Bear in mind also the time printed by the program - about 0.035 seconds on the MI210 - as we will use this as a comparison metric as well.
The timer surrounds the whole `target` region, so this time also includes copying the arrays to and from the GPU.

### The target directive

Let's start by looking at the `target` directive itself.
In the example code [`saxpy_gpu_target.F90`](./Fortran/saxpy_gpu_target.F90), we use the target directive with no additional clauses.
Let's see this directive as it is in the code:

In [ ]:
grep "omp" saxpy_gpu_target.F90

The target directive instructs the compiler to offload the enclosed code to the device.
In our example, this is the subsequent saxpy `do` loop.
With no further instructions or tips given to the compiler on what the code might be doing or how we might want it to run, let's find out how well it can handle the loop for itself:

In [ ]:
make saxpy_gpu_target
./saxpy_gpu_target

Whilst we have successfully offloaded the code to the target device, we can see that we have only created a single team of 256 threads (`1X 256`), and `tripcount:0` shows that the runtime was given no loop to share out.
The answer is still correct, but the run time is about 60 times longer than our baseline run.
We are effectively running the loop in serial on one GPU thread, so there is little point to using the GPU at all.

But of course without adding in any parallelisation instructions to the `target` directive, we should have expected this.
Let's look at what happens if we now create a number of teams, but give them no further directions.

### The teams clause

In the example code [`saxpy_gpu_target_teams.F90`](./Fortran/saxpy_gpu_target_teams.F90), we have added the `teams` clause to the `target` directive.
Let's see the current directive now:

In [ ]:
grep "omp" saxpy_gpu_target_teams.F90

The `teams` clause instructs the directive to create a "league of teams".
The initial thread of each team executes the whole code region, with access to all of the data.
Let's see how the `target teams` directive handles our loop:

In [ ]:
make saxpy_gpu_target_teams
./saxpy_gpu_target_teams

We've successfully offloaded the code, and created a set of 624 teams to run it, but the run time is still about as long as with `target` alone.
So what is going on here?

The teams are indeed running in parallel, but without any instruction that the loop can be split up, each team ends up running the entire loop itself (note `tripcount:0` again)!

You may also notice that instead of the expected values `y(1) 4.000000` and `y(n) 4.000000`, the program prints `********`.
Every team adds `a*x` to the same `y` array, with no protection against teams updating the same elements at the same time.
These race conditions push the values of `y` to 10 or more, which no longer fits the output format, so Fortran prints asterisks instead.

Let's look now at letting the teams share the work properly.

### The distribute clause

The code [`saxpy_gpu_target_teams_distribute.F90`](./Fortran/saxpy_gpu_target_teams_distribute.F90) adds the `distribute` clause to the `target teams` directive. The directive now looks like:

In [ ]:
grep "omp" saxpy_gpu_target_teams_distribute.F90

The `distribute` clause splits up the loop iterations across the available teams, and the initial thread of each team executes its share.
This should solve both problems we had in the previous example, because each iteration is now run exactly once.
Let's see it in action now:

In [ ]:
make saxpy_gpu_target_teams_distribute
./saxpy_gpu_target_teams_distribute

Success!
The results are correct again, and even without the `parallel do` clause the run time (about 0.045 seconds) is close to our baseline with the full directive.

Try changing the array size in [`saxpy_gpu_target_teams_distribute.F90`](./Fortran/saxpy_gpu_target_teams_distribute.F90) (the value of `n` in the `integer,parameter` line), or replacing the kernel with something with more work, and see how the run time and the trace change.
Save the file with Ctrl+S (Cmd+S on a Mac) and run the cell above again.

### Parallel do without a teams distribute clause

As a final experiment, let's see what happens if we include the `parallel do` clause in our `target` directive without the `teams distribute` clauses.
The example [`saxpy_gpu_target_parallel_do.F90`](./Fortran/saxpy_gpu_target_parallel_do.F90) has this already set up for us.

In [ ]:
make saxpy_gpu_target_parallel_do
./saxpy_gpu_target_parallel_do

As expected, only a single team is created (`1X 256`), but the run time (about 0.05 seconds) is still not far from that of the original baseline case.
The `parallel` clause creates multiple threads within a team, and the `do` clause spreads out the work between these threads.
So even without a high teams count, the `parallel do` clause can give good performance.
This can be used to run efficiently with fewer GPU compute units.

## Exploring the split level directives

Finally in this notebook, let's look at different approaches to offloading nested `do` loops to the GPU.

[`saxpy_gpu_collapse.F90`](./Fortran/saxpy_gpu_collapse.F90) contains a nested `do` loop parallelised using the `collapse` clause discussed in the previous lesson.
The code of interest is the loop itself:

```Fortran
!$omp target teams distribute parallel do collapse(2)
do j=1,n
  do i=1,m
    y(i,j) = y(i,j) + a * x(i,j)
  end do
end do
```

As we previously discussed, the `collapse(2)` clause asks the compiler to collapse the 2 subsequent loops into a single one that can be shared out across the GPU.

But now that we've tested the various clauses of this directive, can we not think of another way of splitting this code on the GPU?
Could we not envisage each iteration of the first loop running its own second loop independently, hopefully also in parallel?

The example code [`saxpy_gpu_split_level.F90`](./Fortran/saxpy_gpu_split_level.F90) does exactly that in its revised version of the nested `do` loops:

```Fortran
!$omp target teams distribute
do j=1,n
  !$omp parallel do
  do i=1,m
    y(i,j) = y(i,j) + a * x(i,j)
  end do
end do
```

Here we create a set of teams and distribute the first loop amongst them.
Then the second loop is parallelised within each team.

Which approach do you think will be faster?

Let's run them both now and compare.

In [ ]:
make saxpy_gpu_collapse
./saxpy_gpu_collapse
make saxpy_gpu_split_level
./saxpy_gpu_split_level

There is almost nothing to choose between the two in terms of run time, at least in their original configuration (`n = 1000, m = 1000`).
Notice in the trace that they are launched differently: the collapse version shares out 1,000,000 collapsed iterations (`tripcount:1000000`) across 416 teams, while the split level version gives each of 624 teams whole columns (`tripcount:1000`).

Try changing the size of the input arrays (the values of `n` and `m` in the `integer,parameter` line) in [`saxpy_gpu_collapse.F90`](./Fortran/saxpy_gpu_collapse.F90) and [`saxpy_gpu_split_level.F90`](./Fortran/saxpy_gpu_split_level.F90), save with Ctrl+S (Cmd+S on a Mac) and run the cell above again.
On the MI210, even at `n = 10000, m = 10000` the two stay close (about 0.11 and 0.10 seconds), and the runtime now launches the split level version with 2496 teams of 64 threads.
Which approach is faster depends on the problem size, the compiler and the GPU, so it is always worth measuring.
Set the sizes back to 1000 when you have finished.

Now that we've explored the `target` directive and some of its many possible clauses and configurations, let's move on to memory management directives.
In the next lesson, **Explicit memory management**, we will look at the explicit memory directives.